In [1]:
import glob, os, functools
import numpy as np
import pandas as pd
import SimpleITK as sitk
from util.registration import nrrd_reg_rigid
from util.interpolate import interpolate
from util.bbox import get_bbox_3D
from util.data_util import get_arr_from_nrrd, get_bbox, generate_sitk_obj_from_npy_array
from scipy import ndimage
from SimpleITK.extra import GetArrayFromImage
from scipy import ndimage
import cv2


In [2]:
# Define functiion to normalize the image for training and testing

def normalize_data(patient_id, img, seg, crop_shape, return_type, output_img_dir, 
             output_seg_dir, image_format):

    
    image_arr = sitk.GetArrayFromImage(img)
    image_origin = img.GetOrigin()
    
    label_arr = sitk.GetArrayFromImage(seg)
    
    label_origin = seg.GetOrigin()
    
    # get center. considers all blobs
    bbox = get_bbox(label_arr)
    # returns center point of the label array bounding box
    Z, Y, X = int(bbox[9]), int(bbox[10]), int(bbox[11]) 
    
    #find origin translation from label to image
    origin_dif = tuple(np.subtract(label_origin, image_origin).astype(int))
    
    X_shift, Y_shift, Z_shift = tuple(np.add((X, Y, Z), np.divide(origin_dif, (1, 1, 3)).astype(int)))
    c, y, x = image_arr.shape
    
    ## Get center of mass to center the crop in Y plane
    mask_arr = np.copy(image_arr) 
    mask_arr[mask_arr > -500] = 1
    mask_arr[mask_arr <= -500] = 0
    mask_arr[mask_arr >= -500] = 1 

    centermass = ndimage.measurements.center_of_mass(mask_arr) # z,x,y   
    cpoint = c - crop_shape[2]//2
    centermass = ndimage.measurements.center_of_mass(mask_arr[cpoint, :, :])   
    startx = int(centermass[0] - crop_shape[0]//2)
    starty = int(centermass[1] - crop_shape[1]//2)      
    startz = int(c - crop_shape[2])
    
    
    #-----normalize CT data signals-------
    norm_type = 'np_clip'
    #image_arr[image_arr <= -1024] = -1024
    ## strip skull, skull UHI = ~700
    #image_arr[image_arr > 700] = 0
    ## normalize UHI to 0 - 1, all signlas outside of [0, 1] will be 0;
    if norm_type == 'np_interp':
        image_arr = np.interp(image_arr, [-200, 200], [0, 1])
    elif norm_type == 'np_clip':
        image_arr = np.clip(image_arr, a_min=-175, a_max=275)
        MAX, MIN = image_arr.max(), image_arr.min()
        image_arr = (image_arr - MIN) / (MAX - MIN)
    
    # crop and pad array
    #z_seg_crop = int(crop_shape[2]*0.35)
    if startz < 0:
        image_arr = np.pad(
            image_arr,
            ((abs(startz)//2, abs(startz)//2), (0, 0), (0, 0)), 
            'constant', 
            constant_values=-1024)
        label_arr = np.pad(
            label_arr,
            ((abs(startz)//2, abs(startz)//2), (0, 0), (0, 0)), 
            'constant', 
            constant_values=0)
        image_arr_crop = image_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
        label_arr_crop = label_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
    elif (startx<0) :
        image_arr = np.pad(
            image_arr,
            ((0,0), (abs(startx), abs(startx)), (abs(startx), abs(startx))), 
            'constant', 
            constant_values=0)
        label_arr = np.pad(
            label_arr,
            ((0,0), (abs(startx), abs(startx)), (abs(startx), abs(startx))), 
            'constant', 
            constant_values=0)
        print(startx)
        image_arr_crop = image_arr[0:crop_shape[2], 0:crop_shape[1], 0:crop_shape[0]]
        label_arr_crop = label_arr[0:crop_shape[2], 0:crop_shape[1], 0:crop_shape[0]]
    
    else:
        image_arr_crop = image_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
        label_arr_crop = label_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
    
    # save nrrd
    output_img = output_img_dir + '/' + patient_id + '.' + image_format
    output_seg = output_seg_dir + '/' + patient_id + '.seg.' + image_format
    # save image
    img_sitk = sitk.GetImageFromArray(image_arr_crop)
   
    
    print("final shape to be saved")
    print(img_sitk.GetSize())
    print(img_sitk.GetSpacing())
    
    writer = sitk.ImageFileWriter()
    writer.SetFileName(output_img)
    writer.SetUseCompression(True)
    writer.Execute(img_sitk)
    # save label
    seg_sitk = sitk.GetImageFromArray(label_arr_crop)
   
    #seg_sitk.SetSpacing(seg.GetSpacing())
    #seg_sitk.SetOrigin(seg.GetOrigin())
   
    print("final seg shape to be saved")
    print(seg_sitk.GetSize())
    print(seg_sitk.GetSpacing())
   
    writer = sitk.ImageFileWriter()
    writer.SetFileName(output_seg)
    writer.SetUseCompression(True)
    writer.Execute(seg_sitk)


In [ ]:
#Run this stop providing the data input and output folders as inputs

proj_dir = '../data/raw-data'
image_format = 'nrrd'
img_raw_dir = proj_dir + '/scans'
seg_n_raw_dir = proj_dir + '/expert-segmentations'

img_crop_dir = proj_dir + '/scans-normalized'
seg_n_crop_dir = proj_dir + '/segs-normalized'
   
img_dirs = [i for i in sorted(glob.glob(img_raw_dir + '/*nrrd'))]
seg_n_dirs = [i for i in sorted(glob.glob(seg_n_raw_dir + '/*nrrd'))]
seg_dirs = seg_n_dirs
seg_crop_dir = seg_n_crop_dir
img_ids = []
bad_ids = []
bad_scans = []
count = 0
    
for img_dir in img_dirs:
    img_id = img_dir.split('/')[-1].split('.')[0]
        #print(img_id)
    for seg_dir in seg_dirs:
        seg_id = seg_dir.split('/')[-1].split('.')[0]
      
            #print(seg_id)
        if img_id == seg_id:
            img_ids.append(img_id)
            count += 1
            print(count, img_id)
            # load img and seg
            img = sitk.ReadImage(img_dir, sitk.sitkFloat32)
            seg = sitk.ReadImage(seg_dir, sitk.sitkFloat32)
            z_img = img.GetSize()[2]
            z_seg = seg.GetSize()[2]
            spacing = img.GetSpacing()[-1]
            if z_img < 10:
                print('This is an incomplete scan!')
                bad_scans.append(seg_id)
            else:
                try:
                    print('normalizing')
                    normalize_data(
                            patient_id=img_id,
                            img=img,
                            seg=seg,
                            crop_shape=(512,512,z_seg),
                            return_type='sitk_object',
                            output_img_dir=img_crop_dir,
                            output_seg_dir=seg_crop_dir,
                            image_format=image_format)
                    print('successfully crop!')
                        
                except Exception as e:
                    bad_ids.append(img_id)
                    print(img_id, e)
    print('bad ids:', bad_ids)
    print('incomplete scans:', bad_scans)

